# Compute-cost probe — BIO single-task (E3 word-tagger) vs E4 BIO+CLS multi-task

`idiombert_contributions.md`'s C1 claims BIO is "simpler tagging schema + less compute" than QA-pointer — that claim was **never measured**, only asserted (flagged 2026-06-25). This notebook measures it directly for the two BIO-family systems:

- **BIO single-task** — `run_16_wordpiece_word_tagger.py` (word-level B/I/O tagger, no classifier head)
- **E4 BIO multi-task** — `run_17_bio_cls_joint.py` (BIO span head + CLS idiomaticity head, jointly trained)

Measures, per system: (1) trainable parameter count (head-only diff, since both share the same mBERT backbone — the backbone dominates total params, so this isolates the actual architectural delta), (2) wall-clock seconds/epoch and total training time, same seed (42), same canonical hyperparams as the registered runs in `key_numbers.md`, single run each (timing is not a statistical metric, no 3-seeding needed).

**Does NOT touch canonical artifacts.** Writes to `_timing_bio_s42` / `_timing_e4bio_s42` output dirs, distinct from the canonical `word_tagger_mbert_s42` / `bio_cls_joint_mbert_s42` — those stay untouched. This run's numbers are for the compute-cost question only, not a re-registration of accuracy metrics (ignore exact/overlap/F1 printed here, they're incidental).

**Persistence:** outputs symlinked to Drive, hard-fails before training if not Drive-backed (same gate as every other runner in this repo).

In [ ]:
# 1. Config
REPO_URL = 'https://github.com/JustLetMeBeHello/Idiomator_Research.git'
BRANCH   = 'main'
REPO     = '/content/Idiomator_Research'   # absolute — never use a relative %cd
SEED     = '42'
DRIVE_OUT = '/content/drive/MyDrive/IdiomatorRigor'
print('repo:', REPO, '| seed:', SEED)

In [ ]:
# 2. Clone / refresh repo, pin absolute cwd, kill any nested duplicate clone
import os, subprocess, sys
from pathlib import Path
nested = os.path.join(REPO, 'Idiomator_Research')
if os.path.isdir(nested):
    subprocess.run(['rm', '-rf', nested], check=True)
if os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '--quiet', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '--quiet', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--quiet', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--quiet', '--branch', BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
print('cwd:', os.getcwd())

In [ ]:
# 3. Install deps + confirm GPU
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'Requirements.txt'], check=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU — switch runtime to T4/A100'

In [ ]:
# 4. Mount Drive, make timing-only output dirs, symlink them (Drive-backed,
#    separate namespace from canonical models/word_tagger_mbert_s42 etc.)
from google.colab import drive
drive.mount('/content/drive')
Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)

for name in ('_timing_bio_s42', '_timing_e4bio_s42'):
    drive_path = Path(DRIVE_OUT) / name
    drive_path.mkdir(parents=True, exist_ok=True)
    local_path = Path('models') / name
    if local_path.is_symlink() or local_path.exists():
        if local_path.is_symlink():
            local_path.unlink()
        else:
            raise RuntimeError(f'{local_path} exists and is not a symlink — refusing to overwrite a real dir')
    os.symlink(str(drive_path), str(local_path))
    assert os.path.islink(local_path) and 'drive' in os.readlink(local_path).lower(), \
        f'{local_path} not Drive-backed — refusing to let training write to ephemeral /content'
    print(f'{local_path} -> {os.readlink(local_path)}  (Drive-backed: OK)')

In [ ]:
# 5. Param-count probe — load both model classes (no data needed), count
#    trainable params, and isolate the head-only delta vs the shared mBERT
#    backbone. This answers "is BIO actually fewer params" independent of
#    training time.
import importlib.util, torch

def load_module(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

m16 = load_module('experiments/rigor/run_16_wordpiece_word_tagger.py', 'run16')
m17 = load_module('experiments/rigor/run_17_bio_cls_joint.py', 'run17')

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Both scripts build their model class on top of the same backbone string —
# inspect each module for its model class and instantiate with defaults.
bio_model_cls = [v for k, v in vars(m16).items() if isinstance(v, type) and 'Module' in [c.__name__ for c in v.__mro__]][0]
e4_model_cls  = [v for k, v in vars(m17).items() if isinstance(v, type) and 'Module' in [c.__name__ for c in v.__mro__]][0]

print('BIO single-task model class:', bio_model_cls.__name__)
print('E4 multi-task model class:  ', e4_model_cls.__name__)
print()
print('NOTE: if instantiation below fails (constructor args differ), this cell')
print('is best-effort — fall back to reading param counts from training logs')
print('in cells 6/7 instead (most HF-style trainers print this on startup).')

In [ ]:
# 6. TIMED RUN — BIO single-task (run_16), seed 42, canonical hyperparams
#    (epochs=6, batch=32, lr=3.27e-5 — matches the registered word_tagger_mbert_s42 config).
import time, json

t0 = time.time()
result = subprocess.run([
    sys.executable, 'experiments/rigor/run_16_wordpiece_word_tagger.py',
    '--output_dir', 'models/_timing_bio_s42',
    '--epochs', '6', '--batch_size', '32', '--lr', '3.27e-5',
    '--seed', SEED,
], capture_output=True, text=True)
elapsed_bio = time.time() - t0
print(result.stdout[-3000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-3000:])
print(f'\nBIO single-task wall-clock: {elapsed_bio:.1f}s ({elapsed_bio/60:.2f} min) for 6 epochs')

Path('models/_timing_bio_s42/timing.json').write_text(json.dumps({
    'system': 'bio_single_task_run16', 'seed': int(SEED), 'epochs': 6,
    'wall_clock_seconds': elapsed_bio, 'returncode': result.returncode,
}, indent=2))
print('timing written to Drive: models/_timing_bio_s42/timing.json')

In [ ]:
# 7. TIMED RUN — E4 BIO+CLS multi-task (run_17), seed 42, canonical hyperparams
#    (epochs=7, batch=32, lr=2e-5 — matches the registered bio_cls_joint_mbert_s42 config).
t0 = time.time()
result = subprocess.run([
    sys.executable, 'experiments/rigor/run_17_bio_cls_joint.py',
    '--output_dir', 'models/_timing_e4bio_s42',
    '--epochs', '7', '--batch_size', '32', '--lr', '2e-5',
    '--seed', SEED,
], capture_output=True, text=True)
elapsed_e4 = time.time() - t0
print(result.stdout[-3000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-3000:])
print(f'\nE4 BIO multi-task wall-clock: {elapsed_e4:.1f}s ({elapsed_e4/60:.2f} min) for 7 epochs')

Path('models/_timing_e4bio_s42/timing.json').write_text(json.dumps({
    'system': 'e4_bio_cls_joint_run17', 'seed': int(SEED), 'epochs': 7,
    'wall_clock_seconds': elapsed_e4, 'returncode': result.returncode,
}, indent=2))
print('timing written to Drive: models/_timing_e4bio_s42/timing.json')

In [ ]:
# 8. Persistence readback + per-epoch normalized comparison — reads timing.json
#    back FROM DRIVE (not /content), normalizes to seconds/epoch since the two
#    systems run different epoch counts (6 vs 7) by canonical convention.
import json
rows = []
for name, label in [('_timing_bio_s42', 'BIO single-task (run16)'),
                     ('_timing_e4bio_s42', 'E4 BIO multi-task (run17)')]:
    p = Path(DRIVE_OUT) / name / 'timing.json'
    if not p.exists():
        print(f'{label}: NOT FOUND at {p} — run did not persist')
        continue
    d = json.loads(p.read_text())
    per_epoch = d['wall_clock_seconds'] / d['epochs']
    rows.append((label, d['wall_clock_seconds'], d['epochs'], per_epoch))
    print(f"{label}: total={d['wall_clock_seconds']:.1f}s  epochs={d['epochs']}  sec/epoch={per_epoch:.1f}")

if len(rows) == 2:
    diff_pct = (rows[1][3] - rows[0][3]) / rows[0][3] * 100
    print(f"\nE4 (multi-task) is {diff_pct:+.1f}% sec/epoch vs BIO single-task.")
    print('Reminder: this is ONE seed, no variance estimate — report as a point')
    print('measurement, not a statistically tested claim, unless re-run at 3 seeds.')